# Lesson 04: Feature Detectors/Descriptors (SIFT) — Python (Colab)

This notebook is divided into segments with explanations before each code block. It demonstrates a simplified SIFT-like pipeline: scale-space/DoG keypoints, gradients, descriptor construction, and matching intuition.


## 1. Helpers: Gaussian kernel and convolution

Defines utilities for Gaussian smoothing and 2D convolution, which are building blocks for scale-space construction and gradient computation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gaussian_kernel(s, size=None):
    if size is None: size=int(6*s+1)|1
    r=size//2; y,x=np.mgrid[-r:r+1,-r:r+1]
    k=np.exp(-(x*x+y*y)/(2*s*s)); return k/k.sum()

def conv2(img,k):
    kh,kw=k.shape; ph,pw=kh//2,kw//2
    p=np.pad(img,((ph,ph),(pw,pw)),mode='edge'); out=np.zeros_like(img,np.float32)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]): out[i,j]=np.sum(p[i:i+kh,j:j+kw]*k)
    return out


## 2. Synthetic image and DoG scale-space

Creates a test image with distinctive structures and builds a small Gaussian/DoG pyramid. Difference-of-Gaussians approximates scale-normalized LoG to highlight blob-like structures across scales.


In [ ]:
# Synthetic image
H,W=96,96
yy,xx=np.mgrid[0:H,0:W]
img=np.full((H,W),60,np.float32)
img[20:70,25:40]=200; img[30:45,35:75]=200
img[(yy-70)**2+(xx-70)**2<10*10]=220

# DoG pyramid (one octave, few scales)
sigmas=[1.0,1.6,2.56,4.096]
Gs=[conv2(img, gaussian_kernel(s)) for s in sigmas]
DoGs=[Gs[i+1]-Gs[i] for i in range(len(Gs)-1)]


## 3. Simplified keypoint detection (scale-space extrema)

Searches for local extrema in the middle DoG image compared to neighbors in the same and adjacent scales (a simplified 26-neighborhood idea), producing candidate keypoints.


In [ ]:
# Simplified keypoint: extrema in middle DoG
D=DoGs[1]; prevD, nextD=DoGs[0], DoGs[2]
kps=[]
for i in range(1,H-1):
    for j in range(1,W-1):
        patch=np.stack([prevD[i-1:i+2,j-1:j+2], D[i-1:i+2,j-1:j+2], nextD[i-1:i+2,j-1:j+2]])
        v=D[i,j]
        if v==patch.max() and v>5: kps.append((i,j))
        if v==patch.min() and v<-5: kps.append((i,j))


## 4. Gradients for orientation and description

Computes gradient magnitudes and orientations from a smoothed image. These gradients drive both canonical orientation assignment and histogram-based descriptors in SIFT.


In [ ]:
# Gradients for orientation/descriptor
sobelx=np.array([[-1,0,1],[-2,0,2],[-1,0,1]],np.float32)
sobely=np.array([[-1,-2,-1],[0,0,0],[1,2,1]],np.float32)
Ix=conv2(Gs[1],sobelx); Iy=conv2(Gs[1],sobely)
mag=np.sqrt(Ix*Ix+Iy*Iy); ori=(np.arctan2(Iy,Ix)+2*np.pi)%(2*np.pi)


## 5. Toy SIFT-like descriptor (128-D)

Builds a simplified descriptor by pooling gradient orientation histograms over a 4×4 grid of cells (8 bins each), then normalizing, clamping, and renormalizing — capturing local structure with some robustness.


In [ ]:
def descriptor(i,j):
    # toy 4x4 cells x 8 bins = 128
    desc=np.zeros(128,np.float32)
    for cy in range(4):
        for cx in range(4):
            y0=i-8+cy*4; x0=j-8+cx*4
            hist=np.zeros(8,np.float32)
            for dy in range(4):
                for dx in range(4):
                    y=y0+dy; x=x0+dx
                    if y<0 or x<0 or y>=H or x>=W: continue
                    b=int((ori[y,x]/(2*np.pi))*8)%8
                    hist[b]+=mag[y,x]
            desc[(cy*4+cx)*8:(cy*4+cx+1)*8]=hist
    n=np.linalg.norm(desc)+1e-6; desc/=n
    desc=np.clip(desc,0,0.2); desc/= (np.linalg.norm(desc)+1e-6)
    return desc


## 6. Descriptor comparison and ratio-test intuition

Computes descriptors for a keypoint and compares distances to a similar vs different descriptor. The ratio of best-to-second-best distances indicates how distinctive (unambiguous) a match is.


In [ ]:
# Take a keypoint and create a "second image" rotated/shifted version of its patch conceptually
if not kps: kps=[(40,32),(70,70)]
p=kps[0]; d1=descriptor(p[0],p[1])
# simulate match candidate: similar vs different
d2=d1.copy(); d2[::3]*=0.9; d2/=np.linalg.norm(d2)+1e-6
d3=np.random.rand(128).astype(np.float32); d3/=np.linalg.norm(d3)
def l2(a,b): return np.linalg.norm(a-b)
dist12=l2(d1,d2); dist13=l2(d1,d3)
ratio = min(dist12,dist13)/max(dist12,dist13)


## 7. Visualization

Plots detected keypoints, a DoG response, and the descriptor vector to connect the pipeline stages visually.


In [ ]:
fig,axs=plt.subplots(1,3,figsize=(13,4))
axs[0].imshow(img,cmap='gray'); axs[0].scatter([p[1] for p in kps],[p[0] for p in kps],s=12,c='cyan'); axs[0].set_title('DoG keypoints (simplified)')
axs[1].imshow(DoGs[1],cmap='RdBu_r'); axs[1].set_title('DoG response')
axs[2].bar(np.arange(128),d1,width=1.0); axs[2].set_title('Toy 128-D descriptor')
for a in axs[:2]: a.axis('off')
plt.tight_layout(); plt.show()
print('dist(similar)=%.3f dist(different)=%.3f ratio=%.3f' % (dist12,dist13,ratio))


## Experiments

- Implement full 26-neighbor extrema, subpixel localization/contrast filtering, and a true dominant-orientation histogram.
- Compare with OpenCV `cv2.SIFT_create()` on real images and observe matching behavior.
